# 7차 파인튜닝 (v7) — 사람 라벨(human) 트랙 첫 투입
**v5·v6 연속 기각의 원인 = 소급 매칭의 선택 편향** (모델이 이미 읽는 조각만 GT를 얻음 → 새 정보 0).
v7 재료는 **사람 눈이 정답을 붙인 378줄** — 모델과 독립이라 "모델은 못 읽고 사람은 읽는" 진짜 새 교재.
병합 게이트: 권차 줄 제외·복본 꼬리 제거·숫자줄↔저자GT 오배정 차단·소수점 미노출 분류 GT 금지 (육안 재점검 완료).

**val_human 76줄 = 최초의 편향 없는 현장 평가셋** (배치가 v4 실패 조각 위주라 v4에게 불리한 시험지 — 상승 여력 측정용).
**채택 기준:** human val 상승 + close/low 유지 → 이후 로컬 A/B(골든·autotest 재채점) 통과.
**업로드:** `synth_rec.zip` + `real_rec_data_v3.zip` + `real_rec_data_human.zip` (3개) · GPU(T4) 런타임 · 셀1 후 세션 재시작


In [ ]:
# 1) 설치 — torch 제거(NCCL 충돌 방지) 후 GPU paddle
!pip uninstall -y -q torch torchvision torchaudio 2>/dev/null
!pip install -q paddlepaddle-gpu==3.0.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
!git clone --depth 1 https://github.com/PaddlePaddle/PaddleOCR.git
!pip install -q -r PaddleOCR/requirements.txt
print('✅ 설치 완료 — [런타임 → 세션 다시 시작] 후 셀2부터!')

In [ ]:
# 2) 데이터 업로드(3개) + 층화 배합 — v6: 강화 게이트 field, 가중 3
MIX = {'close': 6, 'low': 2, 'pair': 4, 'human': 3}   # <- 배합 비율 (여기만 바꿔서 재실험)

from google.colab import files
up = files.upload()   # synth_rec.zip, real_rec_data_v3.zip, real_rec_data_human.zip 셋 다 선택
!unzip -oq synth_rec.zip -d PaddleOCR/train_data/
!unzip -oq real_rec_data_v3.zip -d PaddleOCR/train_data/
!unzip -oq real_rec_data_human.zip -d PaddleOCR/train_data/
import io
TAB, NL = chr(9), chr(10)
def read(p): return [l.split(TAB) for l in io.open(p, encoding='utf-8').read().splitlines() if l.strip()]
merged = ['synth_rec/train/' + p + TAB + t for p, t in read('PaddleOCR/train_data/synth_rec/train/rec_gt_train.txt')]
from collections import Counter
used = Counter()
for p, t, g in read('PaddleOCR/train_data/real_rec_data_v3/meta_train.txt'):
    merged += ['real_rec_data_v3/' + p + TAB + t] * MIX.get(g, 1); used[g] += MIX.get(g, 1)
for p, t, g in read('PaddleOCR/train_data/real_rec_data_human/meta_human_train.txt'):
    merged += ['real_rec_data_human/' + p + TAB + t] * MIX.get(g, 1); used[g] += MIX.get(g, 1)
va = {'close': [], 'low': [], 'human': []}
for p, t, g in read('PaddleOCR/train_data/real_rec_data_v3/meta_val.txt'):
    va['close' if g == 'close' else 'low'].append('real_rec_data_v3/' + p + TAB + t)
for p, t, g in read('PaddleOCR/train_data/real_rec_data_human/meta_human_val.txt'):
    va['human'].append('real_rec_data_human/' + p + TAB + t)
io.open('PaddleOCR/train_data/train_v7.txt', 'w', encoding='utf-8').write(NL.join(merged) + NL)
for k, v in va.items():
    io.open('PaddleOCR/train_data/val_' + k + '.txt', 'w', encoding='utf-8').write(NL.join(v) + NL)
io.open('PaddleOCR/train_data/val_all.txt', 'w', encoding='utf-8').write(NL.join(sum(va.values(), [])) + NL)
print('train', len(merged), '줄 · 실전 배합', dict(used))
print('val close', len(va['close']), '/ low', len(va['low']), '/ field', len(va['human']))


In [ ]:
# 3) config·사전학습 모델 자동 탐색
%cd /content
import glob, os
cfgs = glob.glob('PaddleOCR/configs/rec/**/*korean*', recursive=True)
CFG = next((c for c in cfgs if 'v5' in c.lower() and 'mobile' in c.lower()), cfgs[0] if cfgs else None)
print('사용 config:', CFG)
urls = [
 'https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/korean_PP-OCRv5_mobile_rec_pretrained.pdparams',
 'https://paddleocr.bj.bcebos.com/PP-OCRv5/multilingual/korean_PP-OCRv5_mobile_rec_pretrained.pdparams',
]
for u in urls:
    if os.system(f'wget -q {u} -O pretrain.pdparams') == 0 and os.path.getsize('pretrain.pdparams') > 1e6:
        print('사전학습 확보:', u); break

In [ ]:
# 4) 학습 (20 epochs, T4 ~60-80분) — 베스트 선택은 close+low+field 합산 val 기준
%cd /content/PaddleOCR
CFG_REL = CFG.split('PaddleOCR/')[1] if CFG.startswith('PaddleOCR/') else CFG
!python tools/train.py -c {CFG_REL}   -o Global.pretrained_model=/content/pretrain      Global.epoch_num=20      Global.save_model_dir=./output/korean_lowres_v7      Global.eval_batch_step="[0,500]"      Optimizer.lr.learning_rate=0.0001      Train.sampler.first_bs=32      Train.dataset.data_dir=./train_data      Train.dataset.label_file_list=["./train_data/train_v7.txt"]      Eval.dataset.data_dir=./train_data      Eval.dataset.label_file_list=["./train_data/val_all.txt"]


In [ ]:
# 5) close/low/human 3분리 평가 — human이 오르고 close·low가 안 떨어져야 채택
%cd /content/PaddleOCR
for name in ['val_close', 'val_low', 'val_human']:
    print('=====', name, '=====')
    !python tools/eval.py -c {CFG_REL}       -o Global.pretrained_model=./output/korean_lowres_v7/best_accuracy          Eval.dataset.data_dir=./train_data          Eval.dataset.label_file_list=["./train_data/{name}.txt"] 2>/dev/null | grep -E 'acc|norm'


In [ ]:
# 6) 추론 모델로 내보내기 + 다운로드
%cd /content/PaddleOCR
!python tools/export_model.py -c {CFG_REL}   -o Global.pretrained_model=./output/korean_lowres_v7/best_accuracy      Global.save_inference_dir=./korean_lowres_v7_rec_infer
!zip -q -r /content/korean_lowres_v7_rec_infer.zip korean_lowres_v7_rec_infer
from google.colab import files
files.download('/content/korean_lowres_v7_rec_infer.zip')
print('로컬 A/B: daelim_closeup.py --rec_dir korean_lowres_v7_rec_infer (v4 대비 골든·걷기 재채점 후 채택)')
